In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df = pd.read_csv('iris.data.csv')

In [11]:
df.head()

,5.1,3.5,1.4,0.2,Iris-setosa
0,4.9,3.0,1.4,0.2,Iris-setosa
1,4.7,3.2,1.3,0.2,Iris-setosa
2,4.6,3.1,1.5,0.2,Iris-setosa
3,5.0,3.6,1.4,0.2,Iris-setosa
4,5.4,3.9,1.7,0.4,Iris-setosa


In [12]:
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
display(df.head())

,sepal_length,sepal_width,petal_length,petal_width,species
0,4.9,3.0,1.4,0.2,Iris-setosa
1,4.7,3.2,1.3,0.2,Iris-setosa
2,4.6,3.1,1.5,0.2,Iris-setosa
3,5.0,3.6,1.4,0.2,Iris-setosa
4,5.4,3.9,1.7,0.4,Iris-setosa


In [13]:
df.shape

(149, 5)

In [14]:
df['species'].value_counts()

,count
species,
Iris-versicolor,50
Iris-virginica,50
Iris-setosa,49


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X= df.drop('species', axis=1)
y = df['species']

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.33,random_state=42)

In [18]:
from sklearn.neighbors import KNeighborsClassifier

In [19]:
knn = KNeighborsClassifier(n_neighbors=13)

In [20]:
knn.fit(X_train,y_train)

KNeighborsClassifier(n_neighbors=13)

In [21]:
knn.score(X_test,y_test)

0.92

In [22]:
from sklearn.svm import SVC

In [23]:
svm=SVC(gamma = 'auto')

In [24]:
svm.fit(X_train,y_train)

SVC(gamma='auto')

In [25]:
svm.score(X_test,y_test)

0.92

In [26]:
from sklearn.model_selection import GridSearchCV

In [27]:
classifier = GridSearchCV(svm,
                          {
                              'C':[1,5,10,15,20],
                              'kernel':["rbf","linear"],
                          },cv=5,return_train_score=False)

In [28]:
classifier.fit(X,y)

GridSearchCV(cv=5, estimator=SVC(gamma='auto'),
             param_grid={'C': [1, 5, 10, 15, 20], 'kernel': ['rbf', 'linear']})

In [29]:
result = pd.DataFrame(classifier.cv_results_)

In [30]:
result.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002935,0.000624,0.001881,0.000173,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.98,0.016330,1
1,0.002154,0.000085,0.001824,0.000285,1,linear,"{'C': 1, 'kernel': 'linear'}",0.966667,1.0,0.966667,0.966667,1.0,0.98,0.016330,1
2,0.002238,0.000081,0.001784,0.000103,5,rbf,"{'C': 5, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.98,0.016330,1
3,0.002162,0.000223,0.001664,0.000154,5,linear,"{'C': 5, 'kernel': 'linear'}",1.000000,1.0,0.933333,0.966667,1.0,0.98,0.026667,1
4,0.002182,0.000072,0.001629,0.000022,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.98,0.016330,1


In [31]:
result[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,5,rbf,0.980000
3,5,linear,0.980000
4,10,rbf,0.980000
5,10,linear,0.973333
6,15,rbf,0.973333
7,15,linear,0.966667
8,20,rbf,0.966667
9,20,linear,0.966667


In [32]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
    "p": [1, 2]   # Only affects minkowski
}

In [33]:
grid = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [34]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(n_neighbors=13), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan', 'minkowski'],
                         'n_neighbors': [3, 5, 7, 9, 11, 15], 'p': [1, 2],
                         'weights': ['uniform', 'distance']},
             scoring='accuracy')

In [35]:
print(grid.best_params_)

{'metric': 'euclidean', 'n_neighbors': 11, 'p': 1, 'weights': 'uniform'}


In [36]:
print(grid.best_score_)

1.0


In [37]:
best_knn = grid.best_estimator_

In [38]:
y_pred = best_knn.predict(X_test)

In [39]:
test_accuracy=best_knn.score(X_test,y_test)

In [40]:
print(test_accuracy)

0.92


In [41]:
from sklearn.model_selection import RandomizedSearchCV

In [42]:
rcv = RandomizedSearchCV(
    estimator=knn,
    param_distributions=param_grid,  # Changed from param_grid to param_distributions
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    n_iter=10
)

In [43]:
rcv.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=KNeighborsClassifier(n_neighbors=13),
                   n_jobs=-1,
                   param_distributions={'metric': ['euclidean', 'manhattan',
                                                   'minkowski'],
                                        'n_neighbors': [3, 5, 7, 9, 11, 15],
                                        'p': [1, 2],
                                        'weights': ['uniform', 'distance']},
                   scoring='accuracy')

In [44]:
print(rcv.best_params_)

{'weights': 'uniform', 'p': 2, 'n_neighbors': 11, 'metric': 'euclidean'}


In [46]:
best_knn_  = rcv.best_estimator_

In [47]:
y_pred = best_knn_.predict(X_test)

In [48]:
test_accuracy=best_knn_.score(X_test,y_test)

In [49]:
print(test_accuracy)

0.92
